In [33]:
import numpy as np
from pyscf import gto, scf, mp, cc

lno_num = 1
lno_list = [3e-4,1e-4,3e-5,1e-5]
lno_thresh = lno_list[lno_num-1]

####  test H2 monomers ####
a = 2 # bond length in a cluster
d = 5 # distance between each cluster
unit = 'b' # unit of length
na = 2 # size of a cluster (monomer)
nc = 3 # set as integer multiple of monomers
spin = 0 # spin per monomer
frozen = 0 # frozen orbital per monomer
elmt = 'O'
unit = 'B'
basis = '6-31g'
atoms = ""
for n in range(nc*na):
    shift = ((n - n % na) // na) * (d-a)
    atoms += f"{elmt} {n*a+shift:.5f} 0.00000 0.00000 \n"

mol = gto.M(atom=atoms,
            basis=basis,
            verbose=4,
            unit=unit,
            symmetry=0,
            charge=0,
            spin=0,
            max_memory=40000,
            )

mf = scf.RHF(mol).density_fit()
mf.kernel()

mymp = mp.MP2(mf).set_frozen()
mymp.kernel()

mycc = cc.CCSD(mf).set_frozen()
mycc.kernel()

print(f"HF   : {mf.e_tot}")
print(f"MP2  : {mymp.e_tot}")
print(f"CCSD : {mycc.e_tot}")

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-35-generic', version='#35~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue May 26 19:30:42 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Fri Jun 12 17:39:28 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 6
[INPUT] num. electrons = 48
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = B
[INPUT] Symbol           X                Y                Z      uni

In [45]:
mp2dm = mymp.make_rdm1()
# print(mp2dm)

In [ ]:
import jax
jax.config.update("jax_enable_x64", True)

import opt_einsum as oe
def mp2rdm1vir(t2):
    dm_col = 4 * oe.contract('ijac,ijbc->ab', t2, t2, backend='jax')  #\
                  # + oe.contract('ijca,ijcb->ab', t2, t2, backend='jax') )
    dm_exc = 2* oe.contract('ijac,ijcb->ab', t2, t2, backend='jax')  #\
            # + oe.contract('ijca,ijbc->ab', t2, t2, backend='jax')
    dm_vv = dm_col - dm_exc
    return dm_vv

def mp2rdm1occ(t2):
    # for the 1rdm of the full-mp2 wavefuntion
    # subtract HF 1rdm (2 on the diagonal) by this dm_oo
    dm_col = 4 * oe.contract('ikab,jkab->ij', t2, t2, backend='jax')
    dm_exc = 2 * oe.contract('ikab,jkba->ij', t2, t2, backend='jax') 
    dm_oo = dm_col - dm_exc
    return dm_oo

def mp2rdm1vir_frag(t2, p_frag):
    # can't relabel the occ idx when one of them is projected out
    dm_col = 2 * (oe.contract('ik,ijac,kjbc->ab', p_frag, t2, t2, backend='jax')  \
                  + oe.contract('ik,ijca,kjcb->ab', p_frag, t2, t2, backend='jax') )
    dm_exc = oe.contract('ik,ijac,kjcb->ab', p_frag, t2, t2, backend='jax')  \
            + oe.contract('ik,ijca,kjbc->ab', p_frag, t2, t2, backend='jax')
    dm_vv = dm_col - dm_exc
    return dm_vv

def mp2rdm1occ_frag(t2, p_frag):
    dm_col = 4 * oe.contract('kl,ikab,jlab->ij', p_frag, t2, t2, backend='jax')
    dm_exc = 2 * oe.contract('kl,ikab,jlba->ij', p_frag, t2, t2, backend='jax') 
    dm_oo = dm_col - dm_exc
    return dm_oo

In [47]:
mymp.t2.shape

(18, 18, 30, 30)

In [72]:
mymp2dm_vv = mp2rdm1vir(mymp.t2)
nocc = np.count_nonzero(mf.mo_occ)
print((mymp2dm_vv-mp2dm[nocc:,nocc:]).max())

3.469446951953614e-17


In [73]:
mymp2dm_oo = mp2rdm1occ(mymp.t2)
dm_oo = 2 * np.eye(mycc.nocc) - mymp2dm_oo
print((dm_oo-mp2dm[mycc.frozen:nocc,mycc.frozen:nocc]).max())

1.734723475976807e-18


In [74]:
from pyscf import lo
from pyscf.lno.tools import autofrag_iao

orbocc = mf.mo_coeff[:,mycc.frozen:nocc]
iao_coeff = lo.iao.iao(mol, orbocc)
iao_coeff = lo.orth.vec_lowdin(iao_coeff, mf.get_ovlp())
moliao = lo.iao.reference_mol(mol)
frag_lolist = autofrag_iao(moliao)

In [77]:
print(orbocc.shape)
print(iao_coeff.shape)

(54, 18)
(54, 30)


In [78]:
def mo_overlap(s1e, mo1, mo2):
    # <mo1|mo2>
    olp = mo1.T.conj() @ s1e @ mo2
    return olp

In [ ]:
mo2lo = mo_overlap(mf.get_ovlp(), iao_coeff, orbocc)
print(mo2lo.shape)

(30, 18)


In [ ]:
print((mo2lo.T @ mo2lo - np.eye(mo2lo.shape[1])).max())

3.0433988662537104e-14


In [103]:
p_frag_1 = mo2lo[frag_lolist[0],:].T @ mo2lo[frag_lolist[0],]
print(p_frag_1.shape)

(18, 18)


In [89]:
len(frag_lolist)

6

In [104]:
for i,f in enumerate(frag_lolist):
    print(i, f)

0 [0, 1, 2, 3, 4]
1 [5, 6, 7, 8, 9]
2 [10, 11, 12, 13, 14]
3 [15, 16, 17, 18, 19]
4 [20, 21, 22, 23, 24]
5 [25, 26, 27, 28, 29]


In [105]:
def frag_projector(mo2lo, frag_list):
    n_frag = len(frag_list)
    p_frag_list = [None]*n_frag
    
    for i,f in enumerate(frag_list):
        p_frag_list[i] = mo2lo[f,:].T @ mo2lo[f,:]
    
    return p_frag_list

In [106]:
p_frag_list = frag_projector(mo2lo, frag_lolist)

In [108]:
p = np.zeros_like(p_frag_list[0])
for p_frag in p_frag_list:
    p += p_frag

In [110]:
print((p-mo2lo.T @ mo2lo).max())

2.220446049250313e-16


In [111]:
n_frag = len(p_frag_list)
dmvv_frag_list = [None]*n_frag

for i, p_frag in enumerate(p_frag_list):
    dmvv_frag_list[i] = mp2rdm1vir_frag(mymp.t2, p_frag)

In [112]:
d = np.zeros_like(dmvv_frag_list[0])
for i in dmvv_frag_list:
    d += i

In [113]:
print((mymp2dm_vv-d).max())

9.020562075079397e-17


In [115]:
dmoo_frag_list = [None]*n_frag
for i, p_frag in enumerate(p_frag_list):
    dmoo_frag_list[i] = mp2rdm1occ_frag(mymp.t2, p_frag)

d = np.zeros_like(dmoo_frag_list[0])
for i in dmoo_frag_list:
    d += i

print((mymp2dm_oo-d).max())

6.938893903907228e-17


In [118]:
e_occ, u_occ = np.linalg.eigh(dmoo_frag_list[0]) # in asending order
e_occ, u_occ = e_occ[::-1], u_occ[:, ::-1]
print(e_occ.shape)
print(u_occ.shape)

(18,)
(18, 18)


In [119]:
print(e_occ)

[3.10455276e-02 1.64053728e-02 1.17048007e-02 8.28496154e-03
 3.56767931e-03 2.02335439e-03 4.95242487e-05 4.27130447e-05
 4.81296295e-06 3.68619452e-06 1.56752021e-06 8.20490671e-07
 7.96952137e-07 3.94662188e-07 4.69110886e-08 1.67841417e-08
 8.61507957e-09 6.51636846e-09]


In [120]:
thresh_vir = 1e-5
thresh_occ = thresh_vir * 10

In [125]:
e_vir, u_vir = np.linalg.eigh(dmvv_frag_list[0]) # in asending order
e_vir, u_vir = e_vir[::-1], u_vir[:, ::-1]
print(e_vir.shape)
print(u_vir.shape)
print(e_vir)

(30,)
(30, 30)
[3.09092542e-02 1.47543201e-02 1.19109296e-02 7.18282686e-03
 2.89401055e-03 2.01938967e-03 1.65075545e-03 1.04422301e-03
 4.81662062e-04 2.29977212e-04 2.75128048e-05 1.94422801e-05
 3.88938189e-06 2.08489469e-06 1.22822211e-06 1.15381603e-06
 1.14811810e-06 6.69688219e-07 6.11022906e-07 4.89175630e-07
 2.79214251e-07 1.85667626e-07 1.99366140e-08 1.26730823e-08
 5.48182884e-09 4.24870421e-09 2.77685560e-09 1.95924571e-09
 8.01415910e-10 5.42798424e-10]


In [121]:
def mask_no(e, thresh):
    mask = e > thresh
    return mask

In [122]:
mask = e_occ > thresh_occ
u_occ_act = u_occ[:,mask]

In [124]:
no_occ_act = orbocc @ u_occ_act

In [127]:
orbvir = mf.mo_coeff[:,nocc:]
orbvir.shape

(54, 30)

In [128]:
mask = e_vir > thresh_vir
u_vir_act = u_vir[:,mask]
no_vir_act = orbvir @ u_vir_act

In [129]:
print(no_occ_act.shape)
print(no_vir_act.shape)

(54, 6)
(54, 12)


In [130]:
no_act = np.hstack((no_occ_act,no_vir_act))
print(no_act.shape)

(54, 18)


In [ ]:
def plot_density(orbloc, lno_coeff, lno_active, spin_type, idx):
    from pyscf.tools import cubegen
    
    if spin_type == "restricted":
        dm_ctr = orbloc @ orbloc.T
        _ = cubegen.density(mol, f'ctr_density_{idx}.cube', dm_ctr)
        dm_las = lno_coeff[:,lno_active] @ lno_coeff[:,lno_active].T
        _ = cubegen.density(mol, f'las_density_{idx}.cube', dm_las)

    elif spin_type == "unrestricted":
        dm_ctr = orbloc[0] @ orbloc[0].T + orbloc[1] @ orbloc[1].T
        _ = cubegen.density(mol, f'ctr_density_{idx}.cube', dm_ctr)
        dm_las = (lno_coeff[0][:,lno_active] @ lno_coeff[0][:,lno_active].T
                  +lno_coeff[1][:,lno_active] @ lno_coeff[1][:,lno_active].T)
        _ = cubegen.density(mol, f'las_density_{idx}.cube', dm_las)
        
    return None

In [131]:
from pyscf.tools import cubegen
fragctr = iao_coeff[:,frag_lolist[0]]
dm_ctr = fragctr @ fragctr.T
_ = cubegen.density(mol, f'ctr_density_test.cube', dm_ctr)
dm_las = no_act @ no_act.T
_ = cubegen.density(mol, f'las_density_test.cube', dm_las)